In [ ]:
#test_cloud_cover_local
#See also: test_vis_params, test_minmax, test_landsat_histogram

import ee
import geemap

# Initialize Earth Engine
ee.Initialize()

In [ ]:
# ------------------------------------------------------------------
# 1. Define your Area of Interest (AOI)
# ------------------------------------------------------------------
# Example: A polygon around San Francisco (replace with your own)
aoi = ee.Geometry.Polygon(
    [[[-122.55, 37.70],
      [-122.35, 37.70],
      [-122.35, 37.85],
      [-122.55, 37.85],
      [-122.55, 37.70]]])
aoi=ee.Geometry.Polygon( #Margerie
    [[[-137.0999444802902, 59.025089518606336],
      [-137.04128095414362, 59.02602709594012],
      [-137.04305404954465, 59.055910019125456],
      [-137.10176844180705, 59.05497133848375],
      [-137.0999444802902 59.025089518606336]]])

# ------------------------------------------------------------------
# 2. Load a Landsat 8/9 TOA or SR image (TOA has QA_PIXEL)
# ------------------------------------------------------------------
#image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_044034_20210508')  # Example SF
image = ee.Image('LANDSAT/LC08/C02/T1_TOA/LC08_059019_20140903') #Most clear Margerie, but scene has a lot of cloud
# Clip to AOI
image_clipped = image.clip(aoi)

In [ ]:
cloud_cover_wholeimage=image.get('CLOUD_COVER').getInfo()
cloud_cover_landimage=image.get('CLOUD_COVER_LAND').getInfo()
image

In [ ]:
# ------------------------------------------------------------------
# 3. Extract QA_PIXEL band and define cloud/cloud-shadow bits
# ------------------------------------------------------------------
qa = image_clipped.select('QA_PIXEL')

# Bit positions in QA_PIXEL (Landsat 8/9 Collection 2)
# Bit 1: Cloud
# Bit 3: Cloud Shadow
# Bit 4: Snow/Ice (optional)
cloud_bit = 1 << 1   # Cloud
shadow_bit = 1 << 3  # Cloud Shadow
#AKB NOTE: The << operator in Python is the left shift operator. (Shifts binary to the left)

#AKB NOTE: Grok's bit positions are questionable. See https://www.usgs.gov/landsat-missions/landsat-collection-2-quality-assessment-bands
#3	Cloud	
#0 cloud confidence is not high
#1 high confidence cloud
#
#4	Cloud Shadow	
#0 cloud shadow confidence is not high
#1 high confidence cloud shadow
#
#5	Snow	
#0 snow confidence is not high
#1 high confidence snow cover
#
#6 Clear
#
#7 Water

# Create mask: 1 = cloud or shadow, 0 = clear
cloud_shadow_mask = qa.bitwiseAnd(cloud_bit).neq(0).Or(qa.bitwiseAnd(shadow_bit).neq(0))
mask_bit0 = qa.bitwiseAnd(1 << 0).neq(0)
mask_bit1 = qa.bitwiseAnd(1 << 1).neq(0)
mask_bit2 = qa.bitwiseAnd(1 << 2).neq(0)
mask_bit3 = qa.bitwiseAnd(1 << 3).neq(0)
mask_bit4 = qa.bitwiseAnd(1 << 4).neq(0)
mask_bit5 = qa.bitwiseAnd(1 << 5).neq(0)
mask_bit6 = qa.bitwiseAnd(1 << 6).neq(0)
mask_bit7 = qa.bitwiseAnd(1 << 7).neq(0)

# Invert to get clear pixels
clear_mask = cloud_shadow_mask.Not()

# ------------------------------------------------------------------
# 4. Calculate cloud cover percentage over AOI
# ------------------------------------------------------------------
# Count total and clear pixels
pixel_area = ee.Image.pixelArea()

# Total area in AOI
total_area = pixel_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=30,
    maxPixels=1e10
).get('area')

# Clear area
clear_area = pixel_area.updateMask(clear_mask).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=30,
    maxPixels=1e10
).get('area')

# Compute cloud cover %
cloud_cover_percent = ee.Number(1).subtract(
    ee.Number(clear_area).divide(ee.Number(total_area))
).multiply(100)

# ------------------------------------------------------------------
# 5. Get result as a plain Python number
# ------------------------------------------------------------------
cloud_cover = cloud_cover_percent.getInfo()

In [ ]:
ccp=(1-clear_area.getInfo()/total_area.getInfo())*100 #all this type business seems complicated.
ccp

In [ ]:
#One-liner
cloud_cover1 = image.clip(aoi).select('QA_PIXEL') \
    .bitwiseAnd(1<<1).Or(ee.Image().byte().paint(aoi, 1).bitwiseAnd(1<<3)) \
    .reduceRegion(ee.Reducer.mean(), aoi, 30).getInfo()
cloud_cover1
#RESULT: {'QA_PIXEL': 0.01486526717167767} which is not the same as a cloud cover percent...

In [ ]:
#print(f"Cloud + Shadow Cover over AOI: {cloud_cover:.2f}%")
print(f"Whole image: {cloud_cover_wholeimage}%\nLand whole image {cloud_cover_landimage}%\nCloud + Shadow Cover over AOI: {cloud_cover:.2f}% or 1-liner: {cloud_cover1['QA_PIXEL']:.2f}")

In [ ]:
# ------------------------------------------------------------------
# 6. (Optional) Visualize
# ------------------------------------------------------------------
Map = geemap.Map()
Map.centerObject(aoi, 10)

vis_params = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3}
Map.addLayer(image_clipped, vis_params, 'Landsat TOA')
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map.addLayer(cloud_shadow_mask.updateMask(cloud_shadow_mask), {'palette': 'brown'}, 'Cloud+Shadow Mask')
Map.addLayer(mask_bit0.updateMask(mask_bit0), {'palette': 'red'}, '0fill')
Map.addLayer(mask_bit1.updateMask(mask_bit1), {'palette': 'green'}, '1dilated cloud')
Map.addLayer(mask_bit2.updateMask(mask_bit2), {'palette': 'orange'}, '2cirrus')
Map.addLayer(mask_bit3.updateMask(mask_bit3), {'palette': 'red'}, '3cloud') #
Map.addLayer(mask_bit4.updateMask(mask_bit4), {'palette': 'purple'}, '4shadow') #
Map.addLayer(mask_bit5.updateMask(mask_bit5), {'palette': 'lightblue'}, '5snow') #
Map.addLayer(mask_bit6.updateMask(mask_bit6), {'palette': 'white'}, '6clear') #
Map.addLayer(mask_bit7.updateMask(mask_bit7), {'palette': 'blue'}, '7water') #

Map